In [1]:
!pip install gradio tensorflow pillow numpy

In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image
import cv2
import os

# Purane model ko delete karna
if os.path.exists('mnist_cnn_model.keras'):
    os.remove('mnist_cnn_model.keras')

print("Training CNN model (please wait for 5 epochs)...")
mnist = tf.keras.datasets.mnist
(X_train, y_train), (_, _) = mnist.load_data()

# Data normalization and reshaping for CNN
X_train = X_train / 255.0
X_train = np.expand_dims(X_train, -1)

# CNN Architecture
model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Training start hogi aur progress nazar aaye gi
model.fit(X_train, y_train, epochs=5, verbose=1)
model.save('mnist_cnn_model.keras')
print("Model training finished successfully!")

# Prediction Function
def predict_digit(image):
    if image is None:
        return "Please upload an image first!"

    img = Image.fromarray(image.astype('uint8')).convert('L')
    img_np = np.array(img)

    _, thresh = cv2.threshold(img_np, 50, 255, cv2.THRESH_BINARY)
    resized = cv2.resize(thresh, (28, 28), interpolation=cv2.INTER_AREA)

    img_array = resized.astype('float32') / 255.0
    img_array = img_array.reshape(1, 28, 28, 1)

    preds = model.predict(img_array)
    digit = int(np.argmax(preds))
    confidence = float(np.max(preds)) * 100

    return f"Prediction: {digit} (Confidence: {confidence:.2f}%)"

# Gradio App Launch
demo = gr.Interface(
    fn=predict_digit,
    inputs=gr.Image(type="numpy", label="Upload an image of a handwritten number"),
    outputs=gr.Textbox(label="Result"),
    title="🔢 CNN Handwritten Digit Recognition App",
    description="Upload your handwritten number for high confidence prediction."
)

demo.launch(share=True, inline=False)

Training CNN model (please wait for 5 epochs)...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 59s 30ms/step - accuracy: 0.9573 - loss: 0.1389
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 56s 30ms/step - accuracy: 0.9863 - loss: 0.0454
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 81s 30ms/step - accuracy: 0.9904 - loss: 0.0302
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 57s 30ms/step - accuracy: 0.9929 - loss: 0.0226
Epoch 5/5
  19/1875 ━━━━━━━━━━━━━━━━━━━━ 1:25 46ms/step - accuracy: 0.9932 - loss: 0.0122